# Boundary-contract repair review

This is an isolated trial. Production geometry, calibration and RAPID outputs are unchanged.

**Three separate questions:** did the boundary contract pass, are exterior reaches preserved, and are fragment attributes ready for routing? An anchor pass alone is insufficient for the other two.

Run all cells, then change `EXAMPLE` / `STATE`. No ipywidgets dependency.

The viewer now shows **original parent / current / trial** separately. Orange parent links are the selected unit; grey child dots are context, not unit membership. `P` means an original parent ID, `C` a raw child ID, and `H` a new hydraulic ID. All links inside the rectangular viewing window are included. The three panels share zoom ranges.


In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
import geopandas as gpd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src/rapid_tools').exists())
sys.path.insert(0,str(ROOT/'RAPID/src'))
sys.path.insert(0,str(ROOT/'RAPID/investigations'))
from rapid_tools import collapse_creation_review as review
from rapid_tools.anchor_repair_trial import node_mapping
from rapid_tools.boundary_contract_review import plot_comparison as plot, correspondence_table
OUT=ROOT/'RAPID/outputs/boundary_contract_repair_20261008'
results=pd.read_csv(OUT/'full/state_results.csv',dtype={'example':str})
display(results.groupby(['example','status']).size().unstack(fill_value=0))
QUEUE_KIND='anchors'  # 'anchors', 'connectivity', or 'hydraulics'
queue_files={'anchors':'priority_review_queue.csv','connectivity':'replacement_connectivity_review.csv','hydraulics':'hydraulic_review_queue.csv'}
SHOW_REVIEWED=False  # Set True to revisit cases already discussed.
if QUEUE_KIND=='anchors' and (OUT/'manual_review_queue.csv').exists():
    queue_files['anchors']='manual_review_queue.csv'
queue=pd.read_csv(OUT/queue_files[QUEUE_KIND],dtype={'example':str})
if QUEUE_KIND=='anchors' and 'manual_review_status' in queue:
    if not SHOW_REVIEWED:
        queue=queue[queue.manual_review_status.eq('not_reviewed')]
    print(f'{len(queue)} cases shown; visual reviews do not clear automatic identity or hydraulic checks.')
triage_path=OUT/'review_revision/connection_flag_triage.csv'
if QUEUE_KIND=='anchors' and not SHOW_REVIEWED and triage_path.exists():
    triage=pd.read_csv(triage_path,dtype={'example':str})
    assessed=set(zip(triage.example,triage.state))
    queue=queue[~queue.apply(lambda r:(r.example,r.state) in assessed,axis=1)]
    print(f'{len(queue)} cases remain for manual anchor review after automated connection triage.')
display(queue[[c for c in ['example','state','manual_review_status','automatic_review_categories','reason','failed_boundary_checks','unresolved_boundaries','attribute_reason'] if c in queue]])

In [ ]:
EXAMPLE='37'
STATE='S289_unit_143'
TRIAL_STAGE='latest'  # latest uses your reviewed revision when available; 'full' shows the earlier trial
revision_folder=OUT/'review_revision'/EXAMPLE/STATE
if STATE.startswith('island_'):
    revision_folder=OUT/'review_revision/island_boundary_trials'/STATE
folder=revision_folder if TRIAL_STAGE=='latest' and (revision_folder/'trial_links.parquet').exists() else OUT/'full'/EXAMPLE/STATE
print('Trial folder:', folder)
row=review.case_row(EXAMPLE,STATE)
contract=json.loads((folder/'contract.json').read_text())
print('State result:')
display(results[(results.example==EXAMPLE)&(results.state==STATE)].T)
if (folder/'result.json').exists():
    display(json.loads((folder/'result.json').read_text()))
required_for_review=contract.get('reviewed_required_boundary_nodes',contract['boundary_nodes'])
print('Reviewed internal nodes:',contract.get('reviewed_internal_parent_nodes',[]))
print('Accepted provisional / unmatched nodes:',contract.get('reviewed_provisional_parent_nodes',[]))
print('Original automated boundary contract (retained for provenance):')
display(pd.DataFrame(contract['boundaries']))
if (folder/'repair_proposal.json').exists():
    proposal=json.loads((folder/'repair_proposal.json').read_text())
    display(pd.DataFrame(proposal['records']))
# Optional matched-scale close-up; use only one focus namespace at a time.
FOCUS_PARENT_NODE=None    # e.g. 89 for the matching change in 15 / S087_unit_44
FOCUS_CHILD_NODE=None     # e.g. 376 for 15 / S158_unit_191
VIEW_RADIUS_M=80
SHOW_LINK_IDS=True
EXTRA_CHILD_NODES=[]      # e.g. [365,383,412,413] for 15 / S113_unit_189


In [ ]:
frame=pd.read_parquet(row.frame_path)
mapping=node_mapping(frame)
source_links=review.vectors(row.child_links_path,'links')
before=source_links[['id_link','geometry']].merge(frame[['id_link','id_us_node','id_ds_node','u','v']],on='id_link',validate='one_to_one')
assert set(before.id_link)==set(source_links.id_link), 'Source/hydraulic link inventory mismatch'
nodes=review.vectors(row.child_nodes_path,'nodes').copy()
nodes['parent_identity']=nodes.id_node.map(mapping)
if (folder/'trial_links.parquet').exists():
    after=gpd.read_parquet(folder/'trial_links.parquet')
    new_nodes=gpd.read_parquet(folder/'trial_nodes.parquet')
else:
    after=before.copy();new_nodes=nodes.copy()
parent=review.vectors(row.base_links_path,'links')
parent_nodes=review.vectors(row.base_nodes_path,'nodes')
extra=list(EXTRA_CHILD_NODES)+([FOCUS_CHILD_NODE] if FOCUS_CHILD_NODE is not None else [])
display(correspondence_table(nodes,new_nodes,parent_nodes,required_for_review,extra_child_nodes=extra))
fig=plot(before,after,parent,nodes,new_nodes,parent_nodes,required_for_review,
         contract['selected_parent_links'],f'{EXAMPLE} — {STATE}: parent / current / trial',
         focus_parent_node=FOCUS_PARENT_NODE,focus_child_node=FOCUS_CHILD_NODE,radius_m=VIEW_RADIUS_M,
         show_link_ids=SHOW_LINK_IDS)
fig.show(config={'responsive':True})


In [ ]:
if (folder/'validation.json').exists():
    validation=json.loads((folder/'validation.json').read_text())
    display(pd.DataFrame(validation['boundaries']))
    exterior=pd.DataFrame(validation['exterior'])
    display(exterior[~exterior.geometry_exact])
if (folder/'fragment_width_audit.csv').exists():
    widths=pd.read_csv(folder/'fragment_width_audit.csv')
    display(widths[widths.width_status.ne('saved_unsplit_width')])
    if (folder/'fragment_attributes_audit.json').exists():
        print(json.loads((folder/'fragment_attributes_audit.json').read_text())['remaining_requirements'])
if (folder/'validation.json').exists():
    print('Parent versus replacement entry-to-exit reachability (separate from anchoring):')
    routes=pd.DataFrame(validation.get('replacement_connections',[]))
    display(routes)
if (folder/'strict_head_failure.json').exists():
    display(json.loads((folder/'strict_head_failure.json').read_text()))

if (folder/'attribute_checks.json').exists():
    display(json.loads((folder/'attribute_checks.json').read_text()))
    widths=pd.read_csv(folder/'width_depth_checks.csv')
    display(widths[widths.width_status.ne('saved_unsplit_width')])


## Reading the results

- `repaired_anchor_pass`: the trial restores the required identities/cuts and passes boundary incidence and exterior-connection checks within the recorded geometry envelope.
- `unchanged_anchor_pass`: those checks pass without a proposed repair.
- `partial_repair_unresolved` / `unresolved`: inspect failed boundary records. Do not promote these states.
- `blocked_contract`: the selected domain, target evidence or recorded boundary contract disagrees.
- `geometry_exact` is stricter than a connection check. Displaced nodes retain their positions; no silent coordinate snapping is performed.
- Exterior paths may contain multiple child fragments, but cannot pass through an unexplained branching junction or reuse one child reach for distinct parent exterior branches.
- The geometric envelope is four estimated raster steps and is secondary to topology and saved correspondence evidence. It is not a calibrated hydraulic parameter.
- Trial fragment heads explicitly preserve parent exterior heads. Re-trimmed interior widths still need routing-width QC; calibrated K/X and RAPID preparation have not been rerun.
- Existing island aliases inherit their source unit's status. Separate island creation and duplicate-selection decisions are not altered here.

The `connectivity` queue flags changed entry-to-exit relationships in multi-boundary collapses. These are not automatically anchoring errors: the collapse can change interior flow paths. They are kept separate from local boundary checks and require interpretation before full model certification. The `hydraulics` queue separates fixed-exterior-head conflicts from inadequate fragment width support.
